# Challenge 1 - Tic Tac Toe

In this lab you will perform deep learning analysis on a dataset of playing [Tic Tac Toe](https://en.wikipedia.org/wiki/Tic-tac-toe).

There are 9 grids in Tic Tac Toe that are coded as the following picture shows:

![Tic Tac Toe Grids](tttboard.jpg)

In the first 9 columns of the dataset you can find which marks (`x` or `o`) exist in the grids. If there is no mark in a certain grid, it is labeled as `b`. The last column is `class` which tells you whether Player X (who always moves first in Tic Tac Toe) wins in this configuration. Note that when `class` has the value `False`, it means either Player O wins the game or it ends up as a draw.

Follow the steps suggested below to conduct a neural network analysis using Tensorflow and Keras. You will build a deep learning model to predict whether Player X wins the game or not.

## Step 1: Data Engineering

This dataset is almost in the ready-to-use state so you do not need to worry about missing values and so on. Still, some simple data engineering is needed.

1. Read `tic-tac-toe.csv` into a dataframe.
1. Inspect the dataset. Determine if the dataset is reliable by eyeballing the data.
1. Convert the categorical values to numeric in all columns.
1. Separate the inputs and output.
1. Normalize the input data.

In [ ]:
# Load and inspect the Tic Tac Toe dataset.
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

np.random.seed(42)
tf.random.set_seed(42)

df = pd.read_csv("tic-tac-toe.csv")
print("Dataset shape:", df.shape)
print(df.head())
print("\nMissing values by column:\n", df.isna().sum())
print("\nClass distribution:\n", df["class"].value_counts())
print("\nUnique values by column:")
for col in df.columns:
    print(f"{col}: {df[col].unique()}")

# The grid inputs contain categorical marks ('x', 'o', and 'b').
# One-hot encoding avoids inventing an ordering among these three marks.
X = pd.get_dummies(df.drop(columns="class"), dtype=np.float32)

# Encode the binary class as 0/1, whatever its original labels are.
label_encoder = LabelEncoder()
y = label_encoder.fit_transform(df["class"])
print("\nEncoded classes:", dict(enumerate(label_encoder.classes_)))

# One-hot values are already in the [0, 1] range, so no further scaling is needed.
print("Feature matrix:", X.shape, "Target:", y.shape)
print("Normalized feature range:", X.to_numpy().min(), X.to_numpy().max())


## Step 2: Build Neural Network

To build the neural network, you can refer to your own codes you wrote while following the [Deep Learning with Python, TensorFlow, and Keras tutorial](https://www.youtube.com/watch?v=wQ8BIBpya2k) in the lesson. It's pretty similar to what you will be doing in this lab.

1. Split the training and test data.
1. Create a `Sequential` model.
1. Add several layers to your model. Make sure you use ReLU as the activation function for the middle layers. Use Softmax for the output layer because each output has a single lable and all the label probabilities add up to 1.
1. Compile the model using `adam` as the optimizer and `sparse_categorical_crossentropy` as the loss function. For metrics, use `accuracy` for now.
1. Fit the training data.
1. Evaluate your neural network model with the test data.
1. Save your model as `tic-tac-toe.model`.

In [ ]:
# Split the dataset while preserving the approximate class proportions.
X_train, X_test, y_train, y_test = train_test_split(
    X.to_numpy(), y, test_size=0.2, random_state=42, stratify=y
)
print("Train:", X_train.shape, "Test:", X_test.shape)

# Two softmax outputs: one probability per class (0 and 1).
model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(X_train.shape[1],)),
    tf.keras.layers.Dense(64, activation="relu"),
    tf.keras.layers.Dense(32, activation="relu"),
    tf.keras.layers.Dense(2, activation="softmax"),
])

model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)
model.summary()

history = model.fit(
    X_train, y_train, epochs=50, batch_size=32,
    validation_split=0.2, verbose=0
)

baseline_loss, baseline_accuracy = model.evaluate(X_test, y_test, verbose=0)
print(f"Baseline test loss: {baseline_loss:.4f}")
print(f"Baseline test accuracy: {baseline_accuracy:.4f}")

# Modern Keras requires .keras or .h5 for loadable model files;
# the older folder name 'tic-tac-toe.model' in the instructions is not
# supported by model.save() in current Keras versions.
model.save("tic-tac-toe.keras")
print("Model saved as tic-tac-toe.keras")


## Step 3: Make Predictions

Now load your saved model and use it to make predictions on a few random rows in the test dataset. Check if the predictions are correct.

In [ ]:
# Load the model saved in Step 2 and check random test examples.
loaded_model = tf.keras.models.load_model("tic-tac-toe.keras")
rng = np.random.default_rng(42)
sample_indices = rng.choice(len(X_test), size=min(8, len(X_test)), replace=False)

sample_probs = loaded_model.predict(X_test[sample_indices], verbose=0)
sample_preds = np.argmax(sample_probs, axis=1)

results = pd.DataFrame({
    "test_index": sample_indices,
    "actual": label_encoder.inverse_transform(y_test[sample_indices]),
    "predicted": label_encoder.inverse_transform(sample_preds),
    "confidence": sample_probs.max(axis=1).round(3),
    "correct": sample_preds == y_test[sample_indices],
})
print(results.to_string(index=False))
print(f"Correct predictions in sample: {results['correct'].sum()}/{len(results)}")


## Step 4: Improve Your Model

Did your model achieve low loss (<0.1) and high accuracy (>0.95)? If not, try to improve your model.

But how? There are so many things you can play with in Tensorflow and in the next challenge you'll learn about these things. But in this challenge, let's just do a few things to see if they will help.

* Add more layers to your model. If the data are complex you need more layers. But don't use more layers than you need. If adding more layers does not improve the model performance you don't need additional layers.
* Adjust the learning rate when you compile the model. This means you will create a custom `tf.keras.optimizers.Adam` instance where you specify the learning rate you want. Then pass the instance to `model.compile` as the optimizer.
    * `tf.keras.optimizers.Adam` [reference](https://www.tensorflow.org/api_docs/python/tf/keras/optimizers/Adam).
    * Don't worry if you don't understand what the learning rate does. You'll learn about it in the next challenge.
* Adjust the number of epochs when you fit the training data to the model. Your model performance continues to improve as you train more epochs. But eventually it will reach the ceiling and the performance will stay the same.

In [ ]:
# Try a deeper network, a smaller learning rate and more epochs.
# Compare on the same held-out test set as the baseline.
# Validation-based early stopping limits overfitting.
improved_model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(X_train.shape[1],)),
    tf.keras.layers.Dense(128, activation="relu"),
    tf.keras.layers.Dense(64, activation="relu"),
    tf.keras.layers.Dense(32, activation="relu"),
    tf.keras.layers.Dense(2, activation="softmax"),
])

improved_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.0005),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)

callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss", patience=20, restore_best_weights=True
    )
]
improved_history = improved_model.fit(
    X_train, y_train, epochs=200, batch_size=16,
    validation_split=0.2, callbacks=callbacks, verbose=0
)

improved_loss, improved_accuracy = improved_model.evaluate(X_test, y_test, verbose=0)
print(f"Improved model trained for {len(improved_history.history['loss'])} epochs")
print(f"Baseline  - loss: {baseline_loss:.4f}, accuracy: {baseline_accuracy:.4f}")
print(f"Modified  - loss: {improved_loss:.4f}, accuracy: {improved_accuracy:.4f}")
print(f"Accuracy change: {improved_accuracy - baseline_accuracy:+.4f}")
print(f"Loss change: {improved_loss - baseline_loss:+.4f}")
print("Meets targets (loss < 0.1, accuracy > 0.95):",
      improved_loss < 0.1 and improved_accuracy > 0.95)

# Save the modified model separately so the baseline is not overwritten.
improved_model.save("tic-tac-toe-improved.keras")


**Which approach(es) did you find helpful to improve your model performance?**

**Answer:** I experimented with adding a hidden layer, reducing Adam's learning rate to `0.0005`, and allowing up to 200 training epochs with early stopping. The code above reports the change in test loss and accuracy compared with the baseline. **The actual improvement cannot be confirmed until the notebook is run with `tic-tac-toe.csv`.** If the new model has better test accuracy and lower loss, the combination of changes was helpful; if not, the simpler baseline may be preferable. Because the changes were tested together, this experiment does not establish which individual change caused the difference.